In [44]:
import pandas as pd
import requests,io
api_url="https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=af57253c-e838-46da-a1f5-12b43afd75f3&limit=1000&sort=datacreationdate%20desc&format=CSV"

In [59]:
###避開SSL驗證
resp=requests.get(api_url,verify=False)

df = pd.read_csv(io.StringIO(resp.text))

print(df)

Empty DataFrame
Columns: [因受限於資源分配，每日呼叫API的次數不可大於5000次。]
Index: []


c:\Users\USER\Desktop\20260308\HTML\PM25\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'data.moenv.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   site              1000 non-null   str  
 1   county            1000 non-null   str  
 2   pm25              1000 non-null   int64
 3   datacreationdate  1000 non-null   str  
 4   itemunit          1000 non-null   str  
dtypes: int64(1), str(4)
memory usage: 39.2 KB


In [47]:
### 清理資料(去空值)
df.describe()

df.head(5)

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,21,2026-04-19 11:00,μg/m3
1,員林,彰化縣,25,2026-04-19 11:00,μg/m3
2,臺灣大道,臺中市,21,2026-04-19 11:00,μg/m3
3,大城,彰化縣,17,2026-04-19 11:00,μg/m3
4,富貴角,新北市,14,2026-04-19 11:00,μg/m3


### subset 約定重複跟刪除nan的依據

In [48]:
df1=df.drop_duplicates(subset=["site","datacreationdate"]).dropna()
df1

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,21,2026-04-19 11:00,μg/m3
1,員林,彰化縣,25,2026-04-19 11:00,μg/m3
2,臺灣大道,臺中市,21,2026-04-19 11:00,μg/m3
3,大城,彰化縣,17,2026-04-19 11:00,μg/m3
4,富貴角,新北市,14,2026-04-19 11:00,μg/m3
...,...,...,...,...,...
995,線西,彰化縣,22,2026-04-18 23:00,μg/m3
996,彰化,彰化縣,30,2026-04-18 23:00,μg/m3
997,西屯,臺中市,24,2026-04-18 23:00,μg/m3
998,忠明,臺中市,24,2026-04-18 23:00,μg/m3


### 檢查重複資料

In [49]:
df2=df.duplicated(subset=["site","datacreationdate"])
df2

0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Length: 1000, dtype: bool

In [50]:
import sqlite3

In [51]:
#unique 插入資料唯一的約束
sqlstr='''
create table if not exists data(
id integer primary key autoincrement,
site text,
county text,
pm25 integer,	
datacreationdate text,
itemunit text,
unique(site,datacreationdate)
)
'''

In [52]:
conn=sqlite3.connect("pm25.db")
cursor=conn.cursor()
conn,cursor

(<sqlite3.Connection at 0x1b5c995b4c0>, <sqlite3.Cursor at 0x1b5c99500c0>)

In [53]:
cursor.execute(sqlstr)
conn.commit()

### 插入資料
- or ignore 忽略

In [54]:
sqlstr='insert or ignore into data (site,county,pm25,datacreationdate,itemunit)\
    values(?,?,?,?,?)'

In [55]:
df1.values.tolist()

[['林森', '臺南市', 21, '2026-04-19 11:00', 'μg/m3'],
 ['員林', '彰化縣', 25, '2026-04-19 11:00', 'μg/m3'],
 ['臺灣大道', '臺中市', 21, '2026-04-19 11:00', 'μg/m3'],
 ['大城', '彰化縣', 17, '2026-04-19 11:00', 'μg/m3'],
 ['富貴角', '新北市', 14, '2026-04-19 11:00', 'μg/m3'],
 ['麥寮', '雲林縣', 15, '2026-04-19 11:00', 'μg/m3'],
 ['關山', '臺東縣', 15, '2026-04-19 11:00', 'μg/m3'],
 ['馬公', '澎湖縣', 18, '2026-04-19 11:00', 'μg/m3'],
 ['金門', '金門縣', 19, '2026-04-19 11:00', 'μg/m3'],
 ['馬祖', '連江縣', 25, '2026-04-19 11:00', 'μg/m3'],
 ['埔里', '南投縣', 22, '2026-04-19 11:00', 'μg/m3'],
 ['復興', '高雄市', 29, '2026-04-19 11:00', 'μg/m3'],
 ['永和', '新北市', 12, '2026-04-19 11:00', 'μg/m3'],
 ['竹山', '南投縣', 32, '2026-04-19 11:00', 'μg/m3'],
 ['中壢', '桃園市', 17, '2026-04-19 11:00', 'μg/m3'],
 ['三重', '新北市', 11, '2026-04-19 11:00', 'μg/m3'],
 ['冬山', '宜蘭縣', 10, '2026-04-19 11:00', 'μg/m3'],
 ['宜蘭', '宜蘭縣', 11, '2026-04-19 11:00', 'μg/m3'],
 ['陽明', '臺北市', 22, '2026-04-19 11:00', 'μg/m3'],
 ['花蓮', '花蓮縣', 17, '2026-04-19 11:00', 'μg/m3'],
 ['臺東', '臺東縣', 14

In [56]:
cursor.executemany(sqlstr,df1.values.tolist())
conn.commit()

In [57]:
cursor.rowcount

0

In [58]:
conn.close()